# Oriane scientific analysis — original_095

Adapted executable copy of original commit `3c07c3667e` (48 original code cells).
Current corrected STICS/Archimed outputs. The original
0.95 STICS denominator is retained in `original_095`; `physical_100` removes
only that factor. The source-cell numbers in figures refer to the original.

Reference-LAI plots are explicitly labelled. DAE was a calendar step, not emergence age.
Original density facets are nominal treatment labels, not realised density.
OLS definitions, population standard deviations, three-decimal rounding before
species quadrature and the authored radial diagrams are retained.


In [ ]:
from __future__ import annotations
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "1-simul_stics").is_dir())
sys.path.insert(0, str(REPO / '4-analyses/1-code'))
from oriane_input_adapter import load_data
from oriane_analysis_metrics import compute_decomposition, compare_alternate_mixed
NORMALIZATION_MODE = 'original_095'
NORMALIZATION_FACTOR = 0.95
OUTPUT_DIR = REPO / '4-analyses/2-outputs/oriane_notebook_rerun_2026-11-10/original_095'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA = load_data(normalization_factor=NORMALIZATION_FACTOR)
np.random.seed(42)
sns.set_theme(style='whitegrid', font_scale=.85)
plt.rcParams.update({'figure.dpi':100, 'savefig.dpi':200})
def publish_figures(cell, name=None):
    for j, num in enumerate(plt.get_fignums()):
        fig = plt.figure(num)
        for ax in fig.axes:
            for getter, setter in ((ax.get_xlabel, ax.set_xlabel), (ax.get_ylabel, ax.set_ylabel)):
                label = getter()
                label = label.replace('laimax', 'Reference LAI (laimax)')
                label = label.replace('laimax', 'Reference LAI (laimax)') if label == 'laimax' else label
                label = label.replace('DAE', 'Days since first retained tropical date')
                label = label.replace('faPAR STICS', f'STICS RAINT / (0.48 × {NORMALIZATION_FACTOR:g} × SW)')
                label = 'Archimed absorbed PAR / incident PAR' if label in ('faPAR','sum_fapar') else label
                label = label.replace('density', 'Nominal design density')
                setter(label)
            if ax.get_xlabel() == 'Date':
                ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
                ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
            ax.tick_params(axis='both', labelsize=8)
        legends=list(fig.legends)+[ax.get_legend() for ax in fig.axes if ax.get_legend() is not None]
        for legend in legends:
            title=legend.get_title().get_text()
            if title=='density':legend.set_title('Nominal design density')
            if title=='% sorghum':legend.set_title('Sorghum row share (%)')
            for text in legend.get_texts():
                if text.get_text()=='density':text.set_text('Nominal design density')
                if text.get_text()=='% sorghum':text.set_text('Sorghum row share (%)')
        title=(f'Separate Oriane-method extension: {globals().get("cohort", "")}'+chr(10)+NORMALIZATION_MODE
               if cell>=100 else f'Oriane analysis · {NORMALIZATION_MODE} · source cell {cell}')
        fig.suptitle(title, y=.995, fontsize=11)
        fig.subplots_adjust(top=.88, bottom=.13, hspace=.42, wspace=.25)
        stem = Path(name).stem if name else f'cell_{cell:02d}'
        if j: stem += f'_{j}'
        png, pdf = OUTPUT_DIR / (stem+'.png'), OUTPUT_DIR / (stem+'.pdf')
        outside = 0
        limits = []
        for ax in fig.axes:
            x0,x1 = ax.get_xlim(); y0,y1 = ax.get_ylim(); limits.append([float(x0),float(x1),float(y0),float(y1)])
            for collection in ax.collections:
                offsets = np.asarray(collection.get_offsets(), dtype=float)
                if offsets.ndim == 2 and offsets.shape[1] == 2:
                    valid = np.isfinite(offsets).all(axis=1)
                    outside += int(np.sum(valid & ((offsets[:,0]<x0)|(offsets[:,0]>x1)|(offsets[:,1]<y0)|(offsets[:,1]>y1))))
        note=None
        if outside and cell in (40,41,42,43):
            note=fig.text(.5,.015,f'Authored zoom: {outside} points outside axes; full-range companion supplied.',ha='center',fontsize=8)
        fig.savefig(png, dpi=200, bbox_inches='tight'); fig.savefig(pdf, bbox_inches='tight')
        if outside and cell in (40,41,42,43):
            for ax in fig.axes:
                points=[np.asarray(c.get_offsets(),float) for c in ax.collections]
                points=[p[np.isfinite(p).all(axis=1)] for p in points if p.ndim==2 and p.shape[1]==2]
                points=[p for p in points if len(p)]
                if points:
                    p=np.concatenate(points); bound=max(float(p.max())*1.08,ax.get_xlim()[1],ax.get_ylim()[1])
                    ax.set(xlim=(0,bound),ylim=(0,bound))
            if note is not None:note.set_text('Full-range companion: all scatter points shown; original thresholds retained.')
            fig.savefig(OUTPUT_DIR/(stem+'_unclipped.png'),dpi=200,bbox_inches='tight')
            fig.savefig(OUTPUT_DIR/(stem+'_unclipped.pdf'),bbox_inches='tight')
        display(fig)
        plt.close(fig)


In [ ]:
df_stics = DATA['df_stics'].copy()
df_stics

In [ ]:
from datetime import timedelta, datetime
first_retained_date = pd.to_datetime(df_stics.Date).min()
df_stics['step_number'] = (pd.to_datetime(df_stics.Date)-first_retained_date).dt.days
df_stics['group'] = df_stics['Plant']
df_stics['algo'] = df_stics['algorithm']
df_stics


In [ ]:
sns.relplot(
    data=df_stics.assign(Date=pd.to_datetime(df_stics.Date)),
    x="Date", 
    y="faPAR STICS",
    hue="usm",
    row="Plant",
    col="algorithm",
    style="densite",
    kind="line",
    seed=42,
)

'''
g = sns.relplot(
    data=fmri,
    x="timepoint", y="signal",
    hue="event", style="event", col="region",
    height=4, aspect=.7, kind="line"
)
(g.map(plt.axhline, y=0, color=".7", dashes=(2, 1), zorder=0)
  .set_axis_labels("Timepoint", "Percent signal change")
  .set_titles("Region: {col_name} cortex")
  .tight_layout(w_pad=0))
'''
publish_figures(3, None)


In [ ]:
df_doe = DATA['df_doe'].copy()
df_doe

In [ ]:
conc_df = DATA['conc_df'].copy()
conc_df

In [ ]:
conc_df

In [ ]:
merged = pd.merge(conc_df, df_doe, on='usm', validate='many_to_one')

In [ ]:
merged

In [ ]:
merged = DATA['merged'].copy()
assert not merged.duplicated(['usm','Date','group','algo']).any()
assert (merged.step_number == (pd.to_datetime(merged.Date)-first_retained_date).dt.days).all()
merged


In [ ]:
dist = {'low':.2,'middle':.4,'high':.8}
n_rows = {'one':1,'low':2,'middle':4,'high':6}
merged['species'] = merged['group']
merged['DAE'] = merged['step_number']
merged['design'] = merged['design'].str.replace(r'^intercrop ', '', regex=True)
merged['interrow_sorghum'] = merged['interrow_distance_principal'].map(dist)
merged['interrow_maize'] = merged['interrow_distance_secondary'].map(dist)
merged['intrarow'] = merged['intrarow_distance'].map(dist)
merged['n_rows_sorghum'] = merged['n_rows_principal'].map(n_rows)
merged['n_rows_maize'] = merged['n_rows_secondary'].map(n_rows)
merged['% sorghum'] = (100*merged.n_rows_sorghum/(merged.n_rows_sorghum+merged.n_rows_maize)).round(1)
merged['density'] = 1/merged.interrow_sorghum/merged.intrarow
merged['legacy_nominal_density'] = merged['density']
merged['sum_fapar'] = merged['faPAR']
merged


In [ ]:
merged = merged.drop(columns=['species_id','species_principal','species_secondary','group'],errors='ignore')
merged

In [ ]:
simultaneous = merged[merged["sowing_date_latest_crop"] == "same"]

In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR",
    hue="design",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_designs.png'))
publish_figures(13, 'light_designs.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="laimax", 
    y="faPAR",
    hue="design",
    row="species",
    col="algo",
    kind="scatter",
)

plot.savefig(str(OUTPUT_DIR / 'light_designs_LAI.png'))
publish_figures(14, 'light_designs_LAI.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR STICS",
    hue="design",
    row="species",
    col="algo",
    style="density",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_stics_designs_density.png'))
publish_figures(15, 'light_stics_designs_density.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR",
    hue="design",
    row="species",
    col="algo",
    style="density",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_designs_density.png'))
publish_figures(16, 'light_designs_density.png')


In [ ]:
alternate_mixed = compare_alternate_mixed(simultaneous)
display(alternate_mixed.metrics.query('mixed_usm == 1'))
display(alternate_mixed.intercrop_metrics.query('mixed_usm == 1'))


In [ ]:
display(alternate_mixed.metrics.query('mixed_usm == 2'))
display(alternate_mixed.intercrop_metrics.query('mixed_usm == 2'))

In [ ]:
display(alternate_mixed.metrics.query('mixed_usm == 3'))
display(alternate_mixed.intercrop_metrics.query('mixed_usm == 3'))

In [ ]:
simultaneous

In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR",
    hue="usm",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)
publish_figures(21, None)


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR",
    hue="row_orientation",
    row="species",
    col="algo",
    style="density",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_orientation.png'))
publish_figures(22, 'light_orientation.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="faPAR",
    hue="density",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_density.png'))
publish_figures(23, 'light_density.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="laimax",
    hue="usm",
    row="species",
    col="algo",
    kind="line",
    seed=42,
    style="design"
)
publish_figures(24, None)


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="DAE", 
    y="laimax",
    hue="density",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'LAI.png'))
publish_figures(25, 'LAI.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="somupvtsem", 
    y="laimax",
    hue="density",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)
publish_figures(26, None)


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="somupvtsem", 
    y="hauteur",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'height.png'))
publish_figures(27, 'height.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="laimax", 
    y="faPAR",
    hue="density",
    row="species",
    col="algo",
    kind="scatter",
)

plot.savefig(str(OUTPUT_DIR / 'light_density_LAI.png'))
publish_figures(28, 'light_density_LAI.png')


In [ ]:
strips = merged[merged['design'] == 'strips'].copy()
assert len(strips)>0

In [ ]:
plot = sns.relplot(
    data=strips,
    x="DAE", 
    y="sum_fapar",
    hue="sowing_date_latest_crop",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)

plot.savefig(str(OUTPUT_DIR / 'light_sowing_date.png'))
publish_figures(30, 'light_sowing_date.png')


In [ ]:
plot = sns.relplot(
    data=strips,
    x="DAE", 
    y="sum_fapar",
    hue="sowing_date_latest_crop",
    style="row_orientation",
    row="species",
    col="algo",
    kind="line",
    seed=42,
)
publish_figures(31, None)


In [ ]:
decomposition = compute_decomposition(simultaneous, expected_species=('maize','sorghum'))
df = decomposition.predictions
rrmse = decomposition.species_metrics
assert len(rrmse)==108
assert decomposition.max_identity_residual < 1e-12
print('OLS:',decomposition.regression_backend,'; unrounded MSE identity residual:',decomposition.max_identity_residual)


In [ ]:
df_rrmse = decomposition.plot_metrics.copy()
metadata = merged[['usm','design','% sorghum','density','row_orientation']].drop_duplicates()
assert not metadata.duplicated('usm').any()
df_rrmse = df_rrmse.merge(metadata,on='usm',validate='many_to_one')
df_rrmse


In [ ]:
from matplotlib.patches import Wedge

def arrange_sns_plot(plot, xlim=1.6, ylim=1.6):
    for ax in plot.axes.flat:
        ax.axline((0, 0), slope=1, color="grey", linestyle="--")
        ax.set_xlim([0,xlim])
        ax.set_ylim([0,ylim])
        
        wedge = Wedge(
            center=(0, 0),
            r=0.5,
            theta1=0,
            theta2=90,
            alpha=0.2,
            color="grey"
        )
        ax.add_patch(wedge)
    
        wedge = Wedge(
            center=(0, 0),
            r=0.6,
            theta1=0,
            theta2=90,
            alpha=0.2,
            color="grey"
        )
        ax.add_patch(wedge)
    
        wedge = Wedge(
            center=(0, 0),
            r=0.7,
            theta1=0,
            theta2=90,
            alpha=0.2,
            color="grey"
        )
        ax.add_patch(wedge)

In [ ]:
def label_point(x, y, val, algo, plot):
    for ax, algo_value in zip(plot.axes.flat, plot.col_names):

        a = pd.DataFrame({
            "x": x,
            "y": y,
            "val": val,
            "algo": algo,
        })
        
        # Keep only points for this algo
        subset = a[a["algo"] == algo_value]

        for _, point in subset.iterrows():

            if point["x"]**2 + point["y"]**2 >= 0.7**2:
                ax.text(
                    point["x"] + 0.03,
                    point["y"] - 0.02,
                    str(point["val"])[-2:],
                    fontsize=8, clip_on=True
                )

In [ ]:
df_tmp = df_rrmse.drop_duplicates(["usm","algo"])

plot = sns.swarmplot(
    data=df_tmp,
    x="algo", 
    y="RMSE_intercrop",
    hue="design",
    dodge=True
)

plot.axline((0, 0.2), slope=0, color="grey", linestyle="--")

for _, point in df_tmp.iterrows():

            if point["RMSE_intercrop"] >= 0.1:
                print(point["algo"], point["usm"], round(point["RMSE_intercrop"],3))
                
plt.savefig(str(OUTPUT_DIR / 'RMSE.png'), dpi=200)
publish_figures(36, 'RMSE.png')


In [ ]:
plot = sns.swarmplot(
    data=df_tmp,
    x="algo", 
    y="RMSE_intercrop",
    hue="% sorghum",
    dodge=True
)

plot.axline((0, 0.2), slope=0, color="grey", linestyle="--")
                
plt.savefig(str(OUTPUT_DIR / 'RMSE_prop.png'), dpi=200)
publish_figures(37, 'RMSE_prop.png')


In [ ]:
plot = sns.swarmplot(
    data=df_tmp,
    x="algo", 
    y="rRMSE_intercrop",
    hue="density",
    dodge=True
)

for _, point in df_tmp.iterrows():

            if point["algo"] == 'Beer' and point["design"] != 'strips':
                print(point["density"], round(point["rRMSE_intercrop"],3))
                
plt.savefig(str(OUTPUT_DIR / 'RMSE_density.png'), dpi=200)
publish_figures(38, 'RMSE_density.png')


In [ ]:
# Original cell contained historical hand-entered RMSE arrays.
# These are not results of the present run: report current data summaries.
df_tmp.groupby(['algo','density'],observed=True).RMSE_intercrop.agg(['count','mean','std'])


In [ ]:
plot = sns.relplot(
    data=df_rrmse,
    x="RMSEu / std_3D", 
    y="RMSEs / std_3D",
    hue="design",
    row="species",
    col="algo",
    kind="scatter",
    size="% sorghum",
)

arrange_sns_plot(plot)

plot.savefig(str(OUTPUT_DIR / 'coucheney_design_prop.png'))
publish_figures(40, 'coucheney_design_prop.png')


In [ ]:
plot = sns.relplot(
    data=df_rrmse,
    x="RMSEu / std_3D", 
    y="RMSEs / std_3D",
    hue="density",
    row="species",
    col="algo",
    kind="scatter",
)

arrange_sns_plot(plot)

plot.savefig(str(OUTPUT_DIR / 'coucheney_density.png'))
publish_figures(41, 'coucheney_density.png')


In [ ]:
plot = sns.relplot(
    data=df_rrmse,
    x="RMSEu_intercrop / std_3D", 
    y="RMSEs_intercrop / std_3D",
    hue="density",
    col="algo",
    kind="scatter",
)

arrange_sns_plot(plot, 2.4, 2.4)

label_point(df_rrmse["RMSEu_intercrop / std_3D"], df_rrmse["RMSEs_intercrop / std_3D"], df_rrmse.usm, df_rrmse.algo, plot) 

plot.savefig(str(OUTPUT_DIR / 'coucheney_density_avg.png'))
publish_figures(42, 'coucheney_density_avg.png')


In [ ]:
plot = sns.relplot(
    data=df_rrmse,
    x="RMSEu_intercrop / std_3D", 
    y="RMSEs_intercrop / std_3D",
    hue="design",
    col="algo",
    kind="scatter",
    size="% sorghum",
)

arrange_sns_plot(plot, 2.4, 2.4)

label_point(df_rrmse["RMSEu_intercrop / std_3D"], df_rrmse["RMSEs_intercrop / std_3D"], df_rrmse.usm, df_rrmse.algo, plot) 

plot.savefig(str(OUTPUT_DIR / 'coucheney_design_prop_avg.png'), dpi=200)
publish_figures(43, 'coucheney_design_prop_avg.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="faPAR", 
    y="faPAR STICS",
    hue="usm",
    row="species",
    col="algo",
    kind="scatter",
)

for ax in plot.axes.flat:
    ax.axline((0, 0), slope=1, color="k", linestyle="--")

# for i, row_value in enumerate(plot.row_names):
#     for j, col_value in enumerate(plot.col_names):

#         ax = plot.axes[i, j]

#         facet_df = df[
#             (df["species"] == row_value) &
#             (df["algo"] == col_value)
#         ]

#         for curve, curve_df in facet_df.groupby("usm"):
#             curve_df = curve_df.sort_values("sum_fapar")

#             ax.plot(
#                 curve_df["sum_fapar"],
#                 curve_df["regression"],
#                 "--",
#                 alpha=0.8,
#                 color="grey"
#             )
publish_figures(44, None)


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="faPAR", 
    y="faPAR STICS",
    hue="design",
    row="species",
    col="algo",
    size="% sorghum",
    kind="scatter",
)

for ax in plot.axes.flat:
    ax.axline((0, 0), slope=1, color="k", linestyle="--")

plot.savefig(str(OUTPUT_DIR / 'fapar_3d_vs_stics_design.png'))
publish_figures(45, 'fapar_3d_vs_stics_design.png')


In [ ]:
plot = sns.relplot(
    data=simultaneous,
    x="faPAR", 
    y="faPAR STICS",
    hue="density",
    row="species",
    col="algo",
    size="% sorghum",
    kind="scatter",
)

for ax in plot.axes.flat:
    ax.axline((0, 0), slope=1, color="k", linestyle="--")
publish_figures(46, None)


In [ ]:
merged.to_csv(OUTPUT_DIR/'paired_original_scope.csv',index=False)
simultaneous.to_csv(OUTPUT_DIR/'paired_simultaneous.csv',index=False)
df_rrmse.to_csv(OUTPUT_DIR/'error_decomposition_species.csv',index=False)
decomposition.intercrop_metrics.to_csv(OUTPUT_DIR/'error_decomposition_intercrop.csv',index=False)
alternate_mixed.metrics.to_csv(OUTPUT_DIR/'alternate_mixed_species.csv',index=False)
alternate_mixed.intercrop_metrics.to_csv(OUTPUT_DIR/'alternate_mixed_intercrop.csv',index=False)
